# EV-Drone-Detector — Colab (Train + Detect)

SPGNet tabanlı event-camera drone init-detection pipeline'ı. Hücreleri **sırayla** çalıştır:

1. GPU / sürüm doğrula
2. Repo'yu klonla
3. Temel bağımlılıkları kur (spconv hariç — Colab'ın PyTorch'unu bozmamak için)
4. CUDA'yı **otomatik algıla** ve eşleşen spconv tekerleğini kur
5. **Runtime → Restart session** (spconv kurulduktan sonra zorunlu)
6. spconv CUDA doğrulaması (gerçek bir sparse op çalıştırır)
7. Testler
8. Synthetic duman testi (1 epoch)
9. EV-UAV dataset'ini Drive'dan bağla
10. Eğitim
11. Detection + görselleştirme

> **Runtime → Change runtime type → GPU (A100 önerilir).**
>
> Colab (2026) PyTorch'u CUDA 12.4 + Python 3.12 ile gelir. spconv kendi CUDA
> runtime'ını paketler, bu yüzden sadece uygun NVIDIA sürücüsü yeterli.

## 1. GPU / sürüm doğrulama

In [ ]:
!nvidia-smi | head -20
import sys, torch
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda)
print('gpu    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'GPU yok! Runtime -> Change runtime type -> GPU sec.'

## 2. Repo'yu klonla
Private repo ise `https://<TOKEN>@github.com/...` formatını kullan.

In [ ]:
REPO_URL = 'https://github.com/yigitkayabagci/ev-drone-detector.git'
REPO_DIR = '/content/ev-drone-detector'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true
!ls

## 3. Temel bağımlılıkları kur (spconv HARİÇ)
`.[dev]` extra'sı spconv'u **içermez** — böylece Colab'ın hazır PyTorch'u (CUDA 12.4)
korunur. spconv'u bir sonraki hücrede CUDA'ya göre kuracağız.

In [ ]:
%cd /content/ev-drone-detector
!pip install -q -e '.[dev]'
print('Temel bağımlılıklar kuruldu.')

## 4. CUDA'yı algıla ve eşleşen spconv tekerleğini kur
`torch.version.cuda` okunur ve uygun `spconv-cuXXX` tekerleği seçilir.
Colab CUDA 12.4 → `spconv-cu124`. (cu126 / cu120 / cu118 fallback'leri de var.)

In [ ]:
import torch, sys, subprocess

cuda = torch.version.cuda
assert cuda is not None, 'CUDA PyTorch yok — Runtime -> GPU sec.'
major, minor = (int(x) for x in cuda.split('.')[:2])

if major == 11:
    wheel = 'spconv-cu118'
elif major >= 12:
    if (major, minor) >= (12, 6) or major >= 13:
        wheel = 'spconv-cu126'   # 12.6+ / 13.x (ileri uyumluluk)
    elif (major, minor) >= (12, 4):
        wheel = 'spconv-cu124'   # Colab 2026 burada
    else:
        wheel = 'spconv-cu120'   # 12.0 - 12.3
else:
    raise RuntimeError(f'Desteklenmeyen CUDA {cuda} — spconv\'u elle kur.')

print(f'CUDA {cuda} -> {wheel}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'{wheel}>=2.3.6'])
print('\nspconv kuruldu. >>> ŞİMDİ: Runtime -> Restart session <<< sonra 6. hücreden devam et.')

## 5. ⚠️ Runtime'ı yeniden başlat
spconv yeni kurulduysa **Runtime → Restart session** yap. Sonra bu hücreyi
atla ve **6. hücreden** devam et (klonlama/kurulum tekrar gerekmez).

## 6. spconv CUDA doğrulaması
Sadece `import` etmek yetmez (kernel'ler tembel yüklenir). Burada gerçek bir
sparse conv çalıştırıp CUDA kernel'lerinin yüklendiğini **erkenden** doğruluyoruz.

In [ ]:
%cd /content/ev-drone-detector
import torch, spconv
import spconv.pytorch as spc

print('spconv:', spconv.__version__, '| torch:', torch.__version__, '| cuda:', torch.version.cuda)
try:
    if torch.cuda.is_available():
        idx = torch.tensor([[0, 0, 0, 0], [0, 1, 1, 1]], dtype=torch.int32).cuda()
        feat = torch.ones(2, 1, dtype=torch.float32).cuda()
        st = spc.SparseConvTensor(feat, idx, [4, 4, 4], batch_size=1)
        conv = spc.SubMConv3d(1, 8, 3, padding=1).cuda()
        out = conv(st)
        print('✓ spconv CUDA kernel doğrulaması OK — out features:', tuple(out.features.shape))
    else:
        print('CUDA yok — CPU modunda.')
except Exception as e:
    raise RuntimeError(
        'spconv CUDA kernel hatası (muhtemelen CUDA uyumsuzluğu):\n' + str(e) +
        '\n\nDüzeltme: !pip uninstall -y spconv-cu118 spconv-cu120 spconv-cu124 spconv-cu126 '
        '&& 4. hücreyi tekrar çalıştır.'
    ) from e

## 7. Testler — spconv dahil tüm testler pass olmalı

In [ ]:
%cd /content/ev-drone-detector
!python -m pytest tests/ -v

## 8. Synthetic duman testi (1 epoch)
Pipeline uçtan uca çalışıyor mu? `--epochs 1 --val-start-epoch 0` ile hızlı bir
tur atıp `checkpoints/best_iou.pt` + `last.pt` üretiyoruz.

In [ ]:
%cd /content/ev-drone-detector
!python scripts/train.py --config configs/default.yaml --synthetic \
    --epochs 1 --val-start-epoch 0 --num-workers 2 --device cuda:0 2>&1 | tail -30
!ls -la checkpoints

## 9. EV-UAV dataset'ini otomatik indir (paylaşılan Drive klasöründen)
Veri şu **paylaşılan** klasörde: `EV-UAV-dataset/{train,val,test}/*.npz`
(sahibi orijinal yazar; senin My Drive'ında değil). Paylaşılan klasörler düz
`drive.mount` ile `MyDrive` altında görünmediği için, Colab'ın **kimlik
doğrulamalı Drive API**'si ile klasörü ID üzerinden listeleyip doğrudan
`data/{train,val,test}` altına indiriyoruz — mount/shortcut **gerekmez**.

Hücreyi çalıştırınca açılan pencerede, **bu veriye erişimi olan Google
hesabınla** (paylaşımın yapıldığı hesap) izin ver. Dosyalar lokal `/content`
diskine iner → eğitim I/O'su Drive'dan okumaktan daha hızlı. Tekrar
çalıştırınca var olanları atlar (resume).

> Sadece detection deneyeceksen `WANT_SPLITS = ['test']` yapman yeterli.

In [ ]:
# EV-UAV-dataset paylasilan klasor (root) ID — train/val/test alt klasorleri burada
ROOT_FOLDER_ID = '1VIkBFx5Po0KPIFBYOL_appLVie5wgdyi'
WANT_SPLITS = ['train', 'val', 'test']    # sadece inference icin: ['test']
DATA_DIR = '/content/ev-drone-detector/data'

import os
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive = build('drive', 'v3')

def _list_children(folder_id):
    """Bir klasorun tum cocuklarini (sayfalama dahil) dondurur."""
    out, token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        out.extend(resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break
    return out

def _download(file_id, dest):
    req = drive.files().get_media(fileId=file_id)
    with open(dest, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk()

# Root'u listele -> train/val/test alt klasorlerini isimden bul (API listesi
# guvenilir; MCP/arama indeksi paylasilan klasorde eksik olabiliyor).
splits = {c['name']: c['id'] for c in _list_children(ROOT_FOLDER_ID)
          if c['mimeType'].endswith('folder')}
print('Klasordeki splitler:', list(splits))

for split in WANT_SPLITS:
    if split not in splits:
        print(f'  ! "{split}" klasoru yok, atlandi'); continue
    out_dir = os.path.join(DATA_DIR, split)
    os.makedirs(out_dir, exist_ok=True)
    items = [f for f in _list_children(splits[split]) if f['name'].endswith('.npz')]
    have = 0
    for i, f in enumerate(items, 1):
        dest = os.path.join(out_dir, f['name'])
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            have += 1; continue
        _download(f['id'], dest)
        if i % 20 == 0 or i == len(items):
            print(f'  {split}: {i}/{len(items)}')
    print(f'{split}: {len(items)} dosya hazir ({have} zaten vardi)')
print('\nBitti ->', DATA_DIR)

In [ ]:
# Indirilen veriyi say + tek .npz format kontrolu
# Beklenen: evs_norm (N,>=5 = x,y,t,p,label) + ev_loc (N,3)
import glob, numpy as np
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'/content/ev-drone-detector/data/{split}/*.npz'))
    print(f'{split:5s}: {n} .npz')

samples = sorted(glob.glob('/content/ev-drone-detector/data/train/*.npz'))
assert samples, 'train/ bos — 9. hucredeki indirme calisti mi? Google hesabini kontrol et.'
d = np.load(samples[0], allow_pickle=True)
print('\nornek:', samples[0].split('/')[-1])
print('keys    :', list(d.keys()))
print('evs_norm:', d['evs_norm'].shape, d['evs_norm'].dtype)
print('ev_loc  :', d['ev_loc'].shape, d['ev_loc'].dtype)

## 10. Gerçek eğitim
Veri 9. hücrede `data/{train,val,test}`'e indi. Default config: Adam lr=1e-3,
StepLR(10, 0.1), 50 epoch, bs=1. Her epoch sonunda `checkpoints/last.pt`,
validation başlayınca (`val_start_epoch=40`) `checkpoints/best_iou.pt` yazılır.

Uzun sürer; Colab oturumunu açık tut. **Hızlı deneme** için:
`--epochs 5 --val-start-epoch 1`. Checkpoint'leri kalıcı yapmak istersen
`configs/default.yaml`'da `training.save_dir`'i bir Drive yoluna yönlendir.

In [ ]:
%cd /content/ev-drone-detector
# Hızlı: --epochs 5 --val-start-epoch 1   |   Tam: asagidaki gibi
!python scripts/train.py --config configs/default.yaml --device cuda:0

## 11. Detection + görselleştirme
`best_iou.pt` yoksa `last.pt` kullan. `--visualize` ile bbox'lı PNG'ler üretilir.

In [ ]:
%cd /content/ev-drone-detector
import os
CKPT = 'checkpoints/best_iou.pt' if os.path.exists('checkpoints/best_iou.pt') else 'checkpoints/last.pt'
print('checkpoint:', CKPT)
!python scripts/detect.py \
    --config configs/default.yaml \
    --checkpoint {CKPT} \
    --input data/test/ \
    --output detections.json \
    --visualize --vis_dir visualizations \
    --device cuda:0
!ls visualizations | head

In [ ]:
# Ilk birkac gorsellestirmeyi inline goster
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/content/ev-drone-detector/visualizations/*.png'))[:5]:
    print(p)
    display(Image(p))

## 12. (Ops.) Programatik kullanım — tracker'a init box besleme
Her `.npz` için `bbox = [x_min, y_min, x_max, y_max]` + `score`. En yüksek skorlu
kutu downstream tracker'ın init'i olarak kullanılabilir.

In [ ]:
import os, glob
from ev_drone_detector.detection.detector import DroneDetector

CKPT = 'checkpoints/best_iou.pt' if os.path.exists('checkpoints/best_iou.pt') else 'checkpoints/last.pt'
detector = DroneDetector.from_config('configs/default.yaml')
detector.load_weights(CKPT)

for f in sorted(glob.glob('/content/ev-drone-detector/data/test/*.npz'))[:3]:
    dets = detector.detect_from_npz(f)
    name = os.path.basename(f)
    if dets:
        best = max(dets, key=lambda d: d['score'])
        print(f'{name}: init_bbox={best["bbox"]}  score={best["score"]:.3f}')
    else:
        print(f'{name}: (no detection)')